# LeetCode 880 — Decoded String at Index bằng Python

Link đề: https://leetcode.com/problems/decoded-string-at-index/ · Độ khó: **Medium** · Chủ đề: Stack, String

Skeleton LeetCode:

```python
class Solution:
    def decodeAtIndex(self, s: str, k: int) -> str:
        ...
```

Ghi chú: đáp án duy nhất (một ký tự). Điểm cốt lõi: chuỗi giải mã có thể dài tới `~2^63` ký tự, **không thể dựng ra**. Ta chỉ cần **độ dài** sau mỗi bước, rồi **đi ngược** từ cuối chuỗi mã hoá, dùng phép **modulo** để "gấp" `k` về lần lặp đầu tiên. Oracle: **giải mã đúng từng bước theo đề** (dựng băng chữ thật) với chuỗi nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đọc chuỗi mã hoá từng ký tự; chữ cái thì **ghi** lên băng; chữ số `d` thì **lặp cả băng hiện tại** thêm `d − 1` lần (tổng cộng `d` lần); trả về ký tự thứ `k` (đếm từ 1) của băng cuối.
2. Nắm công cụ Python: duyệt ngược `reversed`, `str.isdigit`, phép `%` và `//`, mảng độ dài tiền tố, `bisect`, số nguyên không giới hạn.
3. Đi từ **dựng băng thật (brute force, chỉ cho chuỗi ngắn)** → **tính độ dài rồi duyệt ngược với `k %= size`** → **mảng độ dài tiền tố + nhị phân và gấp `k`** → **đệ quy trên chỉ số**.
4. Hiểu mấu chốt: băng sau chữ số `d` là `d` bản sao của băng trước đó, nên ký tự thứ `k` giống ký tự thứ `((k − 1) mod L) + 1` của băng trước (độ dài `L`).
5. Tránh lỗi kinh điển: dựng chuỗi giải mã (hết bộ nhớ), quên trường hợp `k mod size == 0` (ký tự cuối của băng), nhầm chia `size` cho `d` khi gặp chữ số, đếm `k` từ 0, và duyệt xuôi thay vì ngược.

## 2. Đề bài & ví dụ

Cho chuỗi mã hoá `s`. Để giải mã thành một **băng**, đọc `s` từng ký tự:

- Nếu là **chữ cái**: ghi chữ đó lên băng.
- Nếu là **chữ số `d`**: viết lặp **toàn bộ băng hiện tại** thêm `d − 1` lần nữa (tổng cộng băng được lặp `d` lần).

Cho số nguyên `k`, trả về chữ cái thứ `k` (đếm từ **1**) của chuỗi giải mã.

**Ví dụ 1:** `s = "leet2code3"`, `k = 10` → `"o"` (chuỗi giải mã `leetleetcodeleetleetcodeleetleetcode`).

**Ví dụ 2:** `s = "ha22"`, `k = 5` → `"h"` (chuỗi `hahahaha`).

**Ví dụ 3:** `s = "a2345678999999999999999"`, `k = 1` → `"a"` (chuỗi có `8301530446056247680` chữ `a`).

**Ràng buộc:** `2 <= s.length <= 100`; `s` gồm chữ thường và chữ số `2..9`, bắt đầu bằng chữ cái; `1 <= k <= 10^9`; `k` không vượt độ dài chuỗi giải mã; độ dài giải mã `< 2^63`.

In [1]:
import itertools
import random
import time
from bisect import bisect_left
from typing import List

examples = [(("leet2code3", 10), "o"), (("ha22", 5), "h"), (("a2345678999999999999999", 1), "a")]
for (s, k), want in examples:
    print(f"s = {s!r}, k = {k} -> {want!r}")

s = 'leet2code3', k = 10 -> 'o'
s = 'ha22', k = 5 -> 'h'
s = 'a2345678999999999999999', k = 1 -> 'a'


## 3. Kiến thức nền

### 3.1 Độ dài sau mỗi bước

Gọi `L` là độ dài băng hiện tại. Gặp chữ cái: `L += 1`. Gặp chữ số `d`: `L *= d`. Với `s = "leet2code3"`: `l e e t` → `L = 4`; `2` → `8`; `c o d e` → `12`; `3` → `36`.

In [2]:
def total_length(s):
    L = 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
    return L


print(total_length("leet2code3"), total_length("ha22"), total_length("a2345678999999999999999"))
assert total_length("a2345678999999999999999") == 8301530446056247680 < 2 ** 63

36 8 8301530446056247680


### 3.2 Phép `%` và chỉ số bắt đầu từ 1

Băng sau chữ số `d` là `d` bản sao của băng trước (độ dài `L`). Vị trí `k` (1-indexed) trong băng mới tương ứng vị trí `((k − 1) mod L) + 1` của băng trước. Dạng tương đương dễ dùng khi **duyệt ngược**: đặt `k %= L`; nếu `k == 0` nghĩa là ký tự cuối của băng (vị trí `L`).

In [3]:
L = 4                                                   # băng "leet"
tape = "leet" * 3
for k in (1, 4, 5, 8, 9, 12):
    k1 = (k - 1) % L + 1
    print(f"k = {k:2d}: tape[k] = {tape[k - 1]!r}, tương ứng vị trí {k1} của 'leet' = {'leet'[k1 - 1]!r}, k % L = {k % L}")

k =  1: tape[k] = 'l', tương ứng vị trí 1 của 'leet' = 'l', k % L = 1
k =  4: tape[k] = 't', tương ứng vị trí 4 của 'leet' = 't', k % L = 0
k =  5: tape[k] = 'l', tương ứng vị trí 1 của 'leet' = 'l', k % L = 1
k =  8: tape[k] = 't', tương ứng vị trí 4 của 'leet' = 't', k % L = 0
k =  9: tape[k] = 'l', tương ứng vị trí 1 của 'leet' = 'l', k % L = 1
k = 12: tape[k] = 't', tương ứng vị trí 4 của 'leet' = 't', k % L = 0


### 3.3 Duyệt ngược và độ dài biết trước

Băng cuối có độ dài `L_total`. Ta đi ngược `s`: ký tự cuối là `d` thì băng trước đó có độ dài `L_total // d`; là chữ cái thì băng trước có độ dài `L_total − 1`. Ở mỗi bước ta duy trì `k` (đã `mod` theo độ dài hiện tại) và độ dài hiện tại; gặp chữ cái mà `k == 0` (mod độ dài) hoặc `k == độ dài` thì chữ đó chính là đáp án.

### 3.4 Không bao giờ dựng chuỗi giải mã

Độ dài có thể tới `2^63 ≈ 9.2·10^18`. Chỉ làm việc với **độ dài** (số nguyên).

In [4]:
print("2^63 =", 2 ** 63, "ký tự: nếu mỗi ký tự 1 byte thì cần", 2 ** 63 // 10 ** 18, "exabyte")

2^63 = 9223372036854775808 ký tự: nếu mỗi ký tự 1 byte thì cần 9 exabyte


## 4. Ý tưởng nền tảng / trực giác

`s = "leet2code3"`, `k = 10`. Băng cuối dài `36`, là **3 bản sao** của băng dài `12`: `k = 10` nằm trong bản sao đầu, ứng với vị trí `10` của băng dài `12` (`leetleetcode`). Băng `12` là băng `8` thêm `code`: vị trí `10` là chữ thứ `2` trong `code` → `o`. 

Cách làm: bắt đầu từ băng đầy đủ, **lùi** từng ký tự của `s`:

- Gặp chữ số `d`: băng nhỏ hơn có độ dài `L // d`; `k` giữ nguyên nhưng lấy `mod` theo `L // d`.
- Gặp chữ cái: nó chiếm vị trí **cuối** của băng hiện tại. Nếu `k` đúng bằng vị trí cuối (tức `k % L == 0`) thì đó là đáp án; ngược lại bỏ nó, `L -= 1`.

In [5]:
s, k = "leet2code3", 10
L = total_length(s)
print("băng cuối dài", L)
for i in range(len(s) - 1, -1, -1):
    k %= L
    ch = s[i]
    if ch.isdigit():
        L //= int(ch)
        print(f"i={i} '{ch}': chữ số -> L = {L}, k mod = {k}")
    else:
        if k == 0:
            print(f"i={i} '{ch}': k % L == 0 -> đáp án {ch!r}")
            break
        L -= 1
        print(f"i={i} '{ch}': chữ cái, k = {k} != 0 -> bỏ, L = {L}")

băng cuối dài 36
i=9 '3': chữ số -> L = 12, k mod = 10
i=8 'e': chữ cái, k = 10 != 0 -> bỏ, L = 11
i=7 'd': chữ cái, k = 10 != 0 -> bỏ, L = 10
i=6 'o': k % L == 0 -> đáp án 'o'


## 5. Cách 1 — Brute Force: dựng băng thật

**Ý tưởng:** làm đúng đề. Dựng chuỗi giải mã bằng cách đọc từng ký tự: chữ cái thì nối; chữ số thì nhân băng lên `d` lần. Cắt băng ở độ dài `k` ngay khi đạt (vì các bước sau chỉ **thêm vào cuối**, `k` ký tự đầu không đổi).

In [6]:
def decode_brute(s, k):
    tape = ""
    for ch in s:
        if ch.isdigit():
            tape *= int(ch)                              # lặp cả băng: d bản sao
        else:
            tape += ch
        if len(tape) >= k:
            tape = tape[:k]                              # chỉ cần k ký tự đầu; không bao giờ đổi nữa
            break
    return tape[k - 1]


for (s, k), want in examples[:2]:
    assert decode_brute(s, k) == want
print([decode_brute(s, k) for (s, k), _ in examples])

['o', 'h', 'a']


In [7]:
# Code nộp LeetCode cho Cách 1 (dựng tới k ký tự: ví dụ 3 chạy được nhờ cắt sớm, nhưng k = 10^9 thì tốn ~1GB)
class SolutionBrute:
    def decodeAtIndex(self, s: str, k: int) -> str:
        tape = ""
        for ch in s:
            if ch.isdigit():
                tape *= int(ch)
            else:
                tape += ch
            if len(tape) >= k:
                return tape[k - 1]
        return tape[k - 1]

**Độ phức tạp:** thời gian và bộ nhớ `O(min(k, độ dài giải mã))`: tới `10^9` ký tự — quá lớn. (Nhờ cắt sớm, ví dụ 3 chạy tức thì vì `k = 1`.)

**Nút thắt:** ta dựng chuỗi trong khi chỉ cần **một** ký tự. Độ dài là thứ duy nhất cần nhớ.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính độ dài `L` của băng cuối (chỉ số nguyên).
2. Duyệt ngược `s`. Mỗi bước: `k %= L`. Chữ số `d` → `L //= d`. Chữ cái → nếu `k == 0` trả chữ đó, ngược lại `L -= 1`.

Mỗi ký tự một lần: `O(n)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** độ dài rồi duyệt ngược, **Cách 3** mảng độ dài tiền tố + nhị phân và gấp `k`, **Cách 4** đệ quy trên chỉ số.

### Cách 2 — Độ dài rồi duyệt ngược

**Giải thích từng đoạn:**

1. Duyệt xuôi, tính `L`, **dừng ngay khi `L ≥ k`** (các ký tự sau không ảnh hưởng ký tự thứ `k`) và nhớ chỉ số `end` đã dùng. Việc dừng sớm giữ `L < 10^9 · 9`, nhỏ hơn mọi giới hạn số nguyên.
2. Duyệt ngược từ `end`: `k %= L`; chữ cái và `k == 0` → trả; chữ cái khác → `L -= 1`; chữ số → `L //= d`.

In [8]:
def decode_reverse(s, k):
    L, end = 0, 0
    for i, ch in enumerate(s):
        L = L * int(ch) if ch.isdigit() else L + 1
        if L >= k:                                       # phần còn lại của s không ảnh hưởng ký tự thứ k
            end = i
            break
    for i in range(end, -1, -1):
        ch = s[i]
        k %= L
        if ch.isdigit():
            L //= int(ch)
        else:
            if k == 0:
                return ch
            L -= 1
    raise AssertionError("không thể tới đây nếu k hợp lệ")


for (s, k), want in examples:
    assert decode_reverse(s, k) == want
print([decode_reverse(s, k) for (s, k), _ in examples])

['o', 'h', 'a']


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionReverse:
    def decodeAtIndex(self, s: str, k: int) -> str:
        L, end = 0, 0
        for i, ch in enumerate(s):
            L = L * int(ch) if ch.isdigit() else L + 1
            if L >= k:
                end = i
                break
        for i in range(end, -1, -1):
            ch = s[i]
            k %= L
            if ch.isdigit():
                L //= int(ch)
            else:
                if k == 0:
                    return ch
                L -= 1
        return ""

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Mảng độ dài tiền tố, nhị phân và gấp `k`

**Ý tưởng:** `lens[i]` = độ dài băng sau khi đọc `s[:i+1]` (không giảm). Với `k`: chỉ số nhỏ nhất `i` có `lens[i] ≥ k` (tìm bằng `bisect_left`).

- `s[i]` là chữ cái: nó là ký tự cuối của băng dài `lens[i]`, và vì `i` nhỏ nhất nên `k = lens[i]` ⇒ trả `s[i]`.
- `s[i]` là chữ số: `k` nằm trong `d` bản sao của băng dài `lens[i−1]`; **gấp** `k = (k − 1) % lens[i−1] + 1` rồi lặp lại (chỉ số nhỏ nhất mới `≤ i − 1`).

In [10]:
def decode_prefix(s, k):
    lens, L = [], 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
        lens.append(L)
    while True:
        i = bisect_left(lens, k)                         # chỉ số nhỏ nhất có lens[i] >= k
        ch = s[i]
        if not ch.isdigit():
            return ch                                    # k == lens[i]: ký tự vừa ghi
        k = (k - 1) % lens[i - 1] + 1                    # gấp k về băng trước (độ dài lens[i-1])


for (s, k), want in examples:
    assert decode_prefix(s, k) == want
print([decode_prefix(s, k) for (s, k), _ in examples])

['o', 'h', 'a']


In [11]:
# Code nộp LeetCode cho Cách 3
from bisect import bisect_left


class SolutionPrefix:
    def decodeAtIndex(self, s: str, k: int) -> str:
        lens, L = [], 0
        for ch in s:
            L = L * int(ch) if ch.isdigit() else L + 1
            lens.append(L)
        while True:
            i = bisect_left(lens, k)
            if not s[i].isdigit():
                return s[i]
            k = (k - 1) % lens[i - 1] + 1

**Độ phức tạp:** `O(n)` dựng + mỗi vòng gấp `O(log n)` với tối đa `n` vòng ⇒ `O(n log n)`; bộ nhớ `O(n)`. (Lưu ý `lens` có thể tới `2^63` — Python không tràn; ở ngôn ngữ khác dùng `long`.)

### Cách 4 — Đệ quy trên chỉ số

**Ý tưởng:** `get(i, k)` = ký tự thứ `k` của băng sau khi đọc `s[:i+1]` (cần `lens`). Nếu `s[i]` là chữ cái: `k == lens[i]` ⇒ trả `s[i]`, ngược lại `get(i−1, k)`. Nếu `s[i]` là chữ số: `get(i−1, (k − 1) % lens[i−1] + 1)`.

In [12]:
def decode_recursive(s, k):
    lens, L = [], 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
        lens.append(L)

    def get(i, k):
        ch = s[i]
        if ch.isdigit():
            return get(i - 1, (k - 1) % lens[i - 1] + 1)     # gấp k về lần lặp đầu
        if k == lens[i]:
            return ch                                         # ký tự cuối cùng vừa ghi
        return get(i - 1, k)                                  # k nằm trong phần băng trước đó

    return get(len(s) - 1, k)


for (s, k), want in examples:
    assert decode_recursive(s, k) == want
print([decode_recursive(s, k) for (s, k), _ in examples])

['o', 'h', 'a']


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionRecursive:
    def decodeAtIndex(self, s: str, k: int) -> str:
        lens, L = [], 0
        for ch in s:
            L = L * int(ch) if ch.isdigit() else L + 1
            lens.append(L)

        def get(i, k):
            if s[i].isdigit():
                return get(i - 1, (k - 1) % lens[i - 1] + 1)
            return s[i] if k == lens[i] else get(i - 1, k)

        return get(len(s) - 1, k)

**Độ phức tạp:** `O(n)` thời gian (mỗi lời gọi lùi một chỉ số), `O(n)` ngăn xếp (`n ≤ 100`).

## 8. TRACE từng bước

**Trace Cách 2** với `s = "leet2code3"`, `k = 10`.

Pha 1 (xuôi): `l`→1, `e`→2, `e`→3, `t`→4, `2`→8, `c`→9, `o`→10 (`10 ≥ k`): dừng, `end = 6`, `L = 10`.

Pha 2 (ngược từ `i = 6`):

| `i` | ký tự | `k %= L` | hành động | `L` sau |
|---|---|---|---|---|
| 6 | `o` | `10 % 10 = 0` | chữ cái và `k == 0` → **trả `o`** | — |

Với `k = 9`: pha 1 dừng ở `c` (`L = 9`), pha 2 ở `i = 5`: `9 % 9 = 0` → `c`. Hàm trace:

In [14]:
def trace_reverse(s, k):
    L, end = 0, 0
    print(f"s = {s!r}, k = {k}")
    print("pha 1 (xuôi):")
    for i, ch in enumerate(s):
        L = L * int(ch) if ch.isdigit() else L + 1
        print(f"  i={i} {ch!r}: L = {L}")
        if L >= k:
            end = i
            print(f"  L >= k -> dừng, end = {end}")
            break
    print("pha 2 (ngược):")
    for i in range(end, -1, -1):
        ch = s[i]
        k %= L
        if ch.isdigit():
            L //= int(ch)
            print(f"  i={i} {ch!r}: chữ số -> k mod L = {k}, L = {L}")
        elif k == 0:
            print(f"  i={i} {ch!r}: chữ cái, k % L == 0 -> đáp án")
            return ch
        else:
            L -= 1
            print(f"  i={i} {ch!r}: chữ cái, k = {k} != 0 -> bỏ, L = {L}")


assert trace_reverse("leet2code3", 10) == "o"
print()
assert trace_reverse("ha22", 5) == "h"

s = 'leet2code3', k = 10
pha 1 (xuôi):
  i=0 'l': L = 1
  i=1 'e': L = 2
  i=2 'e': L = 3
  i=3 't': L = 4
  i=4 '2': L = 8
  i=5 'c': L = 9
  i=6 'o': L = 10
  L >= k -> dừng, end = 6
pha 2 (ngược):
  i=6 'o': chữ cái, k % L == 0 -> đáp án

s = 'ha22', k = 5
pha 1 (xuôi):
  i=0 'h': L = 1
  i=1 'a': L = 2
  i=2 '2': L = 4
  i=3 '2': L = 8
  L >= k -> dừng, end = 3
pha 2 (ngược):
  i=3 '2': chữ số -> k mod L = 5, L = 4
  i=2 '2': chữ số -> k mod L = 1, L = 2
  i=1 'a': chữ cái, k = 1 != 0 -> bỏ, L = 1
  i=0 'h': chữ cái, k % L == 0 -> đáp án


**Trace Cách 3** cho `"ha22"`, `k = 5`: `lens = [1, 2, 4, 8]`. `bisect_left(lens, 5) = 3` (`s[3] = '2'`, chữ số) ⇒ `k = (5 − 1) % lens[2] + 1 = 4 % 4 + 1 = 1`. Lặp: `bisect_left(lens, 1) = 0`, `s[0] = 'h'` chữ cái ⇒ trả `h` ✓.

In [15]:
def trace_prefix(s, k):
    lens, L = [], 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
        lens.append(L)
    print("lens =", lens, "| k =", k)
    while True:
        i = bisect_left(lens, k)
        print(f"  bisect_left(lens, {k}) = {i}, s[{i}] = {s[i]!r}")
        if not s[i].isdigit():
            print("  chữ cái ->", s[i])
            return s[i]
        k = (k - 1) % lens[i - 1] + 1
        print(f"  chữ số -> gấp k = (k-1) % {lens[i - 1]} + 1 = {k}")


assert trace_prefix("ha22", 5) == "h"

lens = [1, 2, 4, 8] | k = 5
  bisect_left(lens, 5) = 3, s[3] = '2'
  chữ số -> gấp k = (k-1) % 4 + 1 = 1
  bisect_left(lens, 1) = 0, s[0] = 'h'
  chữ cái -> h


## 9. Code nộp LeetCode

Chọn **Cách 2** (`O(n)` thời gian, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `decodeAtIndex(self, s, k)`:

In [16]:
class Solution:
    def decodeAtIndex(self, s: str, k: int) -> str:
        length, end = 0, 0
        for i, ch in enumerate(s):                         # độ dài băng, dừng khi đã đủ k ký tự
            length = length * int(ch) if ch.isdigit() else length + 1
            if length >= k:
                end = i
                break
        for i in range(end, -1, -1):                       # lùi từng ký tự mã hoá
            ch = s[i]
            k %= length
            if ch.isdigit():
                length //= int(ch)                         # bỏ bước lặp: băng nhỏ lại d lần
            else:
                if k == 0:                                 # k là vị trí cuối của băng hiện tại
                    return ch
                length -= 1
        return ""


print([Solution().decodeAtIndex(s, k) for (s, k), _ in examples])      # ['o', 'h', 'a']
assert all(Solution().decodeAtIndex(s, k) == want for (s, k), want in examples)

['o', 'h', 'a']


## 10. Vì sao đúng

**(a) Cấu trúc băng.** Sau khi đọc `s[:i+1]` băng `T_i` thoả: nếu `s[i]` là chữ cái thì `T_i = T_{i−1} + s[i]`; nếu là chữ số `d` thì `T_i = T_{i−1}` lặp `d` lần. Độ dài: `|T_i| = |T_{i−1}| + 1` hoặc `d · |T_{i−1}|`.

**(b) Ký tự thứ `k` chỉ phụ thuộc tiền tố.** Mọi bước sau chỉ **thêm vào cuối** hoặc **lặp lại** băng, nên `k` ký tự đầu của băng cuối trùng với `k` ký tự đầu của `T_j` với `j` nhỏ nhất có `|T_j| ≥ k`. Vì vậy dừng khi `L ≥ k` ở pha xuôi là hợp lệ, và `L` còn nhỏ (`< 9·k`).

**(c) Bất biến của pha ngược.** Ở bước `i` ta giữ `L = |T_i|` và một số `k` thoả: ký tự cần tìm là ký tự thứ `k'` của `T_i`, trong đó `k' = k` nếu `1 ≤ k ≤ L`, còn nếu `k = 0` thì `k' = L` (vị trí cuối). Phép `k %= L` chuẩn hoá về dạng đó (với `k` bất kỳ dương, `k mod L` bằng `0` đúng khi `k'` là bội của `L`, tức vị trí cuối của một bản sao).

**(d) Bước lùi, chữ số `d`.** `T_i` gồm `d` bản sao của `T_{i−1}` (độ dài `m = L/d`). Ký tự thứ `k'` của `T_i` bằng ký tự thứ `((k' − 1) mod m) + 1` của `T_{i−1}`. Trong mã, sau `L //= d` (được `m`), bước kế tiếp thực hiện `k %= m`: giá trị `k mod m` (với `0` nghĩa vị trí `m`) chính là vị trí đó. Hai cách viết tương đương vì `(k' − 1) mod m + 1 ≡ k' (mod m)` và nằm trong `[1, m]`.

**(e) Bước lùi, chữ cái.** `T_i = T_{i−1} + s[i]` với `|T_i| = L`. Nếu `k' = L` (tức `k = 0` sau `k %= L`) thì ký tự cần tìm chính là `s[i]`; ngược lại `1 ≤ k' ≤ L − 1` nằm trong `T_{i−1}` (độ dài `L − 1`) với cùng `k'`, nên chỉ việc `L -= 1`.

**(f) Cách 3 & 4** dùng cùng ý nhưng với chỉ số 1 (`(k − 1) % m + 1`) tránh trường hợp 0.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **dựng băng thật** trên mọi chuỗi nhỏ và mọi `k` hợp lệ (mục 12D), và khớp nhau trên chuỗi dài với `k` lớn (12E–F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dựng băng (cắt ở `k`) | `O(min(k, L))` | `O(min(k, L))` | oracle; không dùng được cho `k = 10^9` |
| 2. Độ dài rồi duyệt ngược | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Độ dài tiền tố + `bisect` | `O(n log n)` | `O(n)` | rõ ràng |
| 4. Đệ quy chỉ số | `O(n)` | `O(n)` | bám công thức |

Đo thực tế (`n = 100`, `k = 10^9`; cách 1 chỉ chạy cho `k` nhỏ):

In [17]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(50):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / 50)
    return best


def random_valid(rng, n, kmax=10 ** 9, digit_p=0.4):
    """Chuỗi mã hoá dài n, bắt đầu bằng chữ cái, độ dài giải mã < 2^63; k hợp lệ."""
    while True:
        s = [rng.choice("abcdefghijklmnopqrstuvwxyz")]
        L = 1
        for _ in range(n - 1):
            if rng.random() < digit_p:
                d = rng.randint(2, 9)
                if L * d >= 2 ** 63:
                    ch = rng.choice("abcdefghijklmnopqrstuvwxyz")
                    L += 1
                else:
                    ch = str(d)
                    L *= d
            else:
                ch = rng.choice("abcdefghijklmnopqrstuvwxyz")
                L += 1
            s.append(ch)
        return "".join(s), rng.randint(1, min(kmax, L))


rng = random.Random(880)
s_big, k_big = random_valid(rng, 100)
for name, fn in [("reverse", decode_reverse), ("prefix", decode_prefix), ("recursive", decode_recursive)]:
    print(f"n = 100, k = {k_big:>10} {name:10s} {bench(fn, (s_big, k_big)) * 1e6:8.2f} us")
print(f"brute với k = 10^5: {bench(decode_brute, ('abc99999', 10 ** 5), 1) * 1e6:9.1f} us (tăng theo k)")

n = 100, k =  174059320 reverse        5.05 us
n = 100, k =  174059320 prefix         8.21 us
n = 100, k =  174059320 recursive     19.03 us
brute với k = 10^5:      11.8 us (tăng theo k)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi nhỏ và mọi `k`, với oracle **dựng băng thật**, (E) ngẫu nhiên có seed, (F) giới hạn (`n = 100`, `k = 10^9`, độ dài giải mã gần `2^63`).

### A. Ví dụ của đề

In [18]:
def oracle(s, k):
    """Giải mã ĐÚNG đề: dựng băng như list ký tự (chỉ dùng cho chuỗi mà độ dài giải mã nhỏ)."""
    tape = []
    for ch in s:
        if ch.isdigit():
            base = list(tape)
            for _ in range(int(ch) - 1):
                tape.extend(base)                       # lặp băng hiện tại thêm d-1 lần
        else:
            tape.append(ch)
    return tape[k - 1]


IMPLS = {
    "reverse": decode_reverse,
    "prefix": decode_prefix,
    "recursive": decode_recursive,
    "Solution": lambda s, k: Solution().decodeAtIndex(s, k),
    "SolutionReverse": lambda s, k: SolutionReverse().decodeAtIndex(s, k),
    "SolutionPrefix": lambda s, k: SolutionPrefix().decodeAtIndex(s, k),
    "SolutionRecursive": lambda s, k: SolutionRecursive().decodeAtIndex(s, k),
}


def check(s, k, want=None, names=None):
    if want is None:
        want = oracle(s, k)
    for name in (names or IMPLS):
        got = IMPLS[name](s, k)
        assert got == want, (name, s, k, got, want)


for (s, k), want in examples[:2]:
    check(s, k, want)
    assert oracle(s, k) == want
for name in IMPLS:
    assert IMPLS[name]("a2345678999999999999999", 1) == "a"            # ví dụ 3: không thể dựng băng, chỉ kiểm cài đặt nhanh
print("ví dụ 1, 2: oracle dựng băng và", len(IMPLS), "cài đặt đúng; ví dụ 3 (băng 8.3·10^18 ký tự): 7 cài đặt nhanh đều 'a'")

ví dụ 1, 2: oracle dựng băng và 7 cài đặt đúng; ví dụ 3 (băng 8.3·10^18 ký tự): 7 cài đặt nhanh đều 'a'


### B. Edge case

- `k = 1`: luôn là chữ cái đầu `s[0]`.
- `k` = độ dài băng (ký tự cuối).
- Chuỗi kết thúc bằng chữ số (`"ha22"`) và chữ cái (`"abc"`).
- `k` rơi đúng vào ranh giới các lần lặp (bội của độ dài băng trước).
- Nhiều chữ số liên tiếp (`"a2345"`).
- Chữ số ngay sau chữ cái đầu (`"a9"`), `k` bất kỳ từ `1` tới `9`.
- Chuỗi không có chữ số (`"abcd"`).

In [19]:
cases = [
    ("abcd", 1, "a"), ("abcd", 4, "d"), ("a9", 9, "a"), ("a9", 1, "a"), ("ab2", 3, "a"), ("ab2", 4, "b"),
    ("ha22", 1, "h"), ("ha22", 2, "a"), ("ha22", 8, "a"), ("leet2code3", 4, "t"), ("leet2code3", 5, "l"),
    ("leet2code3", 8, "t"), ("leet2code3", 9, "c"), ("leet2code3", 12, "e"), ("leet2code3", 13, "l"), ("leet2code3", 36, "e"),
    ("a2345", 1, "a"), ("a2345", 120, "a"), ("ab34", 12, "b"), ("ab34", 7, "a"),
]
for s, k, want in cases:
    check(s, k, want)
    assert oracle(s, k) == want, (s, k)
print(len(cases), "edge case: oracle dựng băng và mọi cài đặt đúng")

20 edge case: oracle dựng băng và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_zero_returns_anything(s, k):                   # quên điều kiện chữ cái: k == 0 trả luôn s[i] dù là chữ số
    L = total_length(s)
    for i in range(len(s) - 1, -1, -1):
        k %= L
        if k == 0:
            return s[i]                                # SAI: có thể là chữ số
        if s[i].isdigit():
            L //= int(s[i])
        else:
            L -= 1


def bug_div_minus_one(s, k):                           # chia L cho d - 1 thay vì d
    L = total_length(s)
    for i in range(len(s) - 1, -1, -1):
        k %= L
        if s[i].isdigit():
            L //= max(1, int(s[i]) - 1)
        else:
            if k == 0:
                return s[i]
            L -= 1


def bug_zero_indexed(s, k):                            # coi k là 0-indexed
    return decode_reverse(s, k + 1) if k + 1 <= total_length(s) else ""


def bug_no_mod(s, k):                                  # không lấy k %= L: so sánh k == L
    L = total_length(s)
    for i in range(len(s) - 1, -1, -1):
        if s[i].isdigit():
            L //= int(s[i])
        else:
            if k == L:
                return s[i]
            L -= 1
    return ""


def bug_forward(s, k):                                 # duyệt xuôi, trả khi đạt k (không xử lý lặp)
    n = 0
    for ch in s:
        if ch.isdigit():
            continue
        n += 1
        if n == k:
            return ch
    return ""


BUGS = [bug_zero_returns_anything, bug_div_minus_one, bug_zero_indexed, bug_no_mod, bug_forward]
rng = random.Random(3)
tests = []
while len(tests) < 400:
    s, k = random_valid(rng, rng.randint(2, 7), kmax=200, digit_p=0.4)
    if total_length(s) <= 3000:
        tests.append((s, k))
for bug in BUGS:
    bad = next(((s, k) for s, k in tests if bug(s, k) != oracle(s, k)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên s = {bad[0]!r}, k = {bad[1]}: ra {bug(*bad)!r}, đúng {oracle(*bad)!r}")

bug_zero_returns_anything  sai trên s = 'cp59ru', k = 20: ra '5', đúng 'p'
bug_div_minus_one          sai trên s = 'cp59ru', k = 20: ra 'c', đúng 'p'
bug_zero_indexed           sai trên s = 'slp', k = 3: ra '', đúng 'p'
bug_no_mod                 sai trên s = 'cp59ru', k = 20: ra '', đúng 'p'
bug_forward                sai trên s = 'cp59ru', k = 20: ra '', đúng 'p'


### D. Vét cạn mọi chuỗi nhỏ

Hai họ chuỗi (bắt đầu bằng chữ cái): (1) mọi chuỗi độ dài `2..5` trên bảng chữ `{a, b, c, d, 2, 3}`; (2) mọi chuỗi độ dài `6` trên `{a, b, 2, 3}`. Với **mọi** `k` từ `1` tới độ dài giải mã. Oracle: dựng băng thật.

In [21]:
t0 = time.perf_counter()
cnt = 0
fast = ("reverse", "prefix", "recursive", "Solution")
for alphabet, lengths in [("abcd23", range(2, 6)), ("ab23", [6])]:
    for n in lengths:
        for tup in itertools.product(alphabet, repeat=n):
            if tup[0].isdigit():
                continue
            s = "".join(tup)
            L = total_length(s)
            full = None
            for k in range(1, L + 1):
                want = oracle(s, k)
                for name in fast:
                    assert IMPLS[name](s, k) == want, (name, s, k)
                cnt += 1
print(cnt, "bộ (chuỗi, k): 4 cài đặt khớp oracle dựng băng;", round(time.perf_counter() - t0, 1), "s")

115250 bộ (chuỗi, k): 4 cài đặt khớp oracle dựng băng; 0.8 s


### E. Ngẫu nhiên có seed

Chuỗi dài tới `12`, độ dài giải mã tới `~10^5` (oracle còn dựng được), `k` ngẫu nhiên; và cho mọi cài đặt kể cả cách 1.

In [22]:
rng = random.Random(20240880)
cnt = 0
while cnt < 1500:
    s, k = random_valid(rng, rng.randint(2, 12), kmax=10 ** 5, digit_p=rng.choice([0.2, 0.4, 0.6]))
    if total_length(s) > 60000:
        continue
    want = oracle(s, k)
    for name in IMPLS:
        assert IMPLS[name](s, k) == want, (name, s, k)
    assert decode_brute(s, k) == want and SolutionBrute().decodeAtIndex(s, k) == want
    cnt += 1
print(cnt, "ca ngẫu nhiên: 7 cài đặt nhanh + 2 bản brute khớp oracle")

1500 ca ngẫu nhiên: 7 cài đặt nhanh + 2 bản brute khớp oracle


### F. Giới hạn

`n = 100`, `k` tới `10^9`, độ dài giải mã gần `2^63`. Không có oracle dựng băng: so các cài đặt nhanh với nhau, kiểm tra **tính nhất quán** (ký tự thứ `k` và thứ `k + L_prefix` trùng khi băng lặp), và các trường hợp biên.

In [23]:
rng = random.Random(500)
fast = ["reverse", "prefix", "recursive", "Solution", "SolutionReverse", "SolutionPrefix", "SolutionRecursive"]
cnt = 0
for _ in range(2000):
    s, k = random_valid(rng, rng.randint(30, 100), kmax=10 ** 9, digit_p=rng.choice([0.3, 0.6, 0.9]))
    res = {name: IMPLS[name](s, k) for name in fast}
    assert len(set(res.values())) == 1, (s, k, res)
    cnt += 1
print(cnt, "ca lớn (n tới 100, k tới 10^9): 7 cài đặt cùng đáp án")

s = "a2345678999999999999999"
L = total_length(s)
print("độ dài giải mã của ví dụ 3:", L, "< 2^63:", L < 2 ** 63)
assert all(Solution().decodeAtIndex(s, k) == "a" for k in (1, 2, 10 ** 9, 123456789))
# tính nhất quán: băng sau chữ số cuối là d bản sao, nên ký tự thứ k trùng ký tự thứ k + m với m = độ dài trước khi nhân
t = "x" + "abcdefg2"
m = total_length(t[:-1])
assert all(Solution().decodeAtIndex(t, k) == Solution().decodeAtIndex(t, k + m) for k in range(1, m + 1))
print("biên: k = 1 luôn là s[0]:", all(Solution().decodeAtIndex(random_valid(rng, 50)[0], 1) for _ in range(50)))

2000 ca lớn (n tới 100, k tới 10^9): 7 cài đặt cùng đáp án
độ dài giải mã của ví dụ 3: 8301530446056247680 < 2^63: True
biên: k = 1 luôn là s[0]: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dựng chuỗi giải mã

Độ dài giải mã có thể là `8·10^18`.

In [24]:
s = "a2345678999999999999999"
print("độ dài giải mã:", total_length(s))
try:
    tape = "a"
    for ch in s:
        tape = tape * int(ch) if ch.isdigit() else tape + ch
        if len(tape) > 10 ** 6:
            raise MemoryError("băng vượt 10^6 ký tự: dừng thử nghiệm")
except MemoryError as e:
    print("MemoryError (giả lập):", e)

độ dài giải mã: 8301530446056247680
MemoryError (giả lập): băng vượt 10^6 ký tự: dừng thử nghiệm


**Vì sao sai:** không thể lưu `8·10^18` ký tự. **Đúng:** chỉ theo dõi **độ dài**, duyệt ngược.

### Lỗi 2: quên điều kiện "chữ cái" khi `k % L == 0`

`k == 0` ở một chữ số **không** có nghĩa tìm được đáp án.

In [25]:
bad = next((s, k) for s, k in tests if bug_zero_returns_anything(s, k) != oracle(s, k))
print(bad, "->", repr(bug_zero_returns_anything(*bad)), "| đúng:", repr(oracle(*bad)))

('cp59ru', 20) -> '5' | đúng: 'p'


**Vì sao sai:** sau `k %= L` mà `k == 0` ở một chữ số, ta chỉ cần tiếp tục lùi (băng nhỏ lại, `k` vẫn mang nghĩa "vị trí cuối" của băng nhỏ). **Đúng:** chỉ trả khi `s[i]` là **chữ cái**.

### Lỗi 3: chia `L` cho `d − 1`

Chữ số `d` lặp băng thành `d` bản (không phải `d − 1`).

In [26]:
bad = next((s, k) for s, k in tests if bug_div_minus_one(s, k) != oracle(s, k))
print(bad, "->", repr(bug_div_minus_one(*bad)), "| đúng:", repr(oracle(*bad)))

('cp59ru', 20) -> 'c' | đúng: 'p'


**Vì sao sai:** đề nói lặp **thêm** `d − 1` lần, tổng cộng `d` bản sao. **Đúng:** `L //= d`.

### Lỗi 4: `k` bắt đầu từ 0

Đề dùng chỉ số **1**.

In [27]:
print("k tính từ 0 trên 'abcd', k = 1:", repr(bug_zero_indexed("abcd", 1)), "| đúng:", repr(Solution().decodeAtIndex("abcd", 1)))
assert bug_zero_indexed("abcd", 1) == "b" and Solution().decodeAtIndex("abcd", 1) == "a"

k tính từ 0 trên 'abcd', k = 1: 'b' | đúng: 'a'


**Vì sao sai:** lệch một vị trí. **Đúng:** `k` 1-indexed; dùng `k %= L` với `0` ↔ vị trí cuối.

### Lỗi 5: duyệt xuôi, bỏ qua các lần lặp

In [28]:
bad = next((s, k) for s, k in tests if bug_forward(s, k) != oracle(s, k))
print(bad, "->", repr(bug_forward(*bad)), "| đúng:", repr(oracle(*bad)))

('cp59ru', 20) -> '' | đúng: 'p'


**Vì sao sai:** ký tự thứ `k` nằm trong các bản sao lặp, không phải trong `s` theo thứ tự. **Đúng:** duyệt **ngược** và gấp `k`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi/dãy được xây bằng thao tác nhân bản, độ dài khổng lồ, hỏi một vị trí" → **không dựng**; lưu **độ dài**, rồi **truy ngược** dùng modulo để gấp vị trí về bản gốc.

**Mẹo:**

1. Tính độ dài trước, dừng khi đã `≥ k`.
2. Duyệt ngược; chữ số → chia; chữ cái → kiểm tra `k % L == 0`.
3. `k % L == 0` nghĩa "ký tự cuối của băng", đừng nhầm với "không tồn tại".
4. Với chỉ số 1: `(k - 1) % m + 1` tránh trường hợp 0.
5. Oracle: dựng băng thật trên chuỗi nhỏ.

**Họ bài cùng pattern:** Kth Symbol in Grammar (779), Find K-th Bit in Nth Binary String (1545), Remove All Occurrences..., Decode String (394), Shifting Letters (848), Elimination Game.

## 15. Biến thể

1. **Lấy cả đoạn** `[l, r]` của chuỗi giải mã (mỗi ký tự qua hàm `decodeAtIndex`).
2. **Đếm** số lần một chữ cái xuất hiện trong chuỗi giải mã (không dựng).
3. **Độ dài giải mã** (đã dùng `total_length`).
4. **Kth Symbol in Grammar (779):** cấu trúc đệ quy tương tự (nhân đôi).
5. **Chữ số `0`/`1`** (xử lý riêng).

Cài biến thể 1 (đoạn) và 2 (đếm), đối chiếu oracle:

In [29]:
def decode_range(s, l, r):
    """Chuỗi con [l, r] (1-indexed, gồm hai đầu) của chuỗi giải mã."""
    return "".join(Solution().decodeAtIndex(s, k) for k in range(l, r + 1))


def count_letter(s, letter):
    """Đếm số lần `letter` xuất hiện trong chuỗi giải mã, không dựng: cnt nhân theo chữ số, cộng theo chữ cái."""
    cnt = 0
    for ch in s:
        if ch.isdigit():
            cnt *= int(ch)
        elif ch == letter:
            cnt += 1
    return cnt


def decode_all(s):
    tape = []
    for ch in s:
        if ch.isdigit():
            tape = tape * int(ch)
        else:
            tape.append(ch)
    return "".join(tape)


rng = random.Random(15)
cnt = 0
while cnt < 300:
    s, _ = random_valid(rng, rng.randint(2, 9), kmax=10, digit_p=0.4)
    if total_length(s) > 2000:
        continue
    full = decode_all(s)
    l = rng.randint(1, len(full))
    r = rng.randint(l, len(full))
    assert decode_range(s, l, r) == full[l - 1:r]
    for letter in "abc":
        assert count_letter(s, letter) == full.count(letter)
    cnt += 1
print("decode_range và count_letter khớp chuỗi giải mã thật trên 300 ca; ví dụ 3: số chữ 'a' =", count_letter("a2345678999999999999999", "a"))

decode_range và count_letter khớp chuỗi giải mã thật trên 300 ca; ví dụ 3: số chữ 'a' = 8301530446056247680


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng băng | mô phỏng, cắt ở `k` | `O(min(k, L))` | `O(min(k, L))` | oracle, `k` nhỏ |
| 2. Độ dài + duyệt ngược | `k %= L`, chữ số chia, chữ cái trừ | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Độ dài tiền tố + `bisect` | tìm tiền tố đầu tiên đủ `k`, gấp | `O(n log n)` | `O(n)` | rõ ràng |
| 4. Đệ quy chỉ số | `get(i, k)` | `O(n)` | `O(n)` | bám công thức |

Cả bốn khớp oracle dựng băng trên mọi chuỗi nhỏ và mọi `k` (mục 12D) và khớp nhau trên chuỗi dài `k` tới `10^9`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(total_length("ab3"), total_length("a2b3"), Solution().decodeAtIndex("ab3", 5), Solution().decodeAtIndex("a2b3", 9))

6 9 a b


### Bài 2 (tính tay)

`s = "xy2z3"`. (a) Độ dài băng sau mỗi ký tự? (b) Chuỗi giải mã đầy đủ? (c) Ký tự thứ `13`? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện bước lùi:

In [31]:
def reverse_todo(s, k):
    L = 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1        # độ dài băng cuối (cho phép lớn, Python không tràn)
    for i in range(len(s) - 1, -1, -1):
        k %= L
        pass            # TODO 1: nếu s[i] là chữ số: L //= int(s[i])
        pass            # TODO 2: nếu là chữ cái: k == 0 thì trả s[i], ngược lại L -= 1
    return ""


# Khi làm xong, bỏ comment để kiểm tra:
# assert [reverse_todo(s, k) for (s, k), _ in examples] == ["o", "h", "a"]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mảng độ dài tiền tố + `bisect_left`**:

In [32]:
class MySolution:
    def decodeAtIndex(self, s, k):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().decodeAtIndex(s, k) == w for (s, k), w in examples)

### Bài 5 (mini challenge): vị trí đầu tiên trong `s` của ký tự được chọn

Viết `source_index(s, k)` trả **chỉ số `i` trong chuỗi mã hoá** `s` của chữ cái sinh ra ký tự thứ `k` (nghĩa là `s[i] == decodeAtIndex(s, k)`, và `i` là vị trí chữ cái trong `s` đã "ghi" ký tự đó).

In [33]:
def source_index(s, k):
    pass            # TODO


# Khi làm xong:
# assert source_index("leet2code3", 10) == 6

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `L` tăng 1 ở chữ cái, nhân `d` ở chữ số.
- Mức 2: `"ab3"` → `2` rồi `6`; `"a2b3"` → `1`, `2`, `3`, `9`.
- Mức 3: kết quả `6 9 a b`: `"ab3"` giải mã `ababab`, ký tự thứ 5 là `a`; `"a2b3"` giải mã `aabaabaab`, ký tự thứ 9 là `b`.

**Bài 2**

- Mức 1: `x`→1, `y`→2, `2`→4, `z`→5, `3`→15.
- Mức 2: băng sau `y2` là `xyxy`, sau `z` là `xyxyz`.
- Mức 3: chuỗi `xyxyzxyxyzxyxyz`; ký tự thứ 13 là `x` (13 = 10 + 3 → vị trí 3 của `xyxyz` là `x`).

**Bài 3**

- Mức 1: chữ số: `L //= int(s[i])`.
- Mức 2: chữ cái: `if k == 0: return s[i]`.
- Mức 3: ngược lại `L -= 1`.

**Bài 4**

- Mức 1: `lens` không giảm.
- Mức 2: `i = bisect_left(lens, k)`.
- Mức 3: chữ số → `k = (k - 1) % lens[i-1] + 1`, chữ cái → trả.

**Bài 5**

- Mức 1: tương tự Cách 3 nhưng trả `i` thay vì `s[i]`.
- Mức 2: khi gặp chữ cái ở `bisect_left` thì chính `i` đó.
- Mức 3: chữ số → gấp `k` rồi lặp.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
assert (total_length("ab3"), total_length("a2b3"), Solution().decodeAtIndex("ab3", 5), Solution().decodeAtIndex("a2b3", 9)) == (6, 9, "a", "b")
assert decode_all("ab3") == "ababab" and decode_all("a2b3") == "aabaabaab"
print("Bài 1 OK: 6 9 a b")

Bài 1 OK: 6 9 a b


### Bài 2

(a) `1, 2, 4, 5, 15`. (b) `xyxyzxyxyzxyxyz`. (c) ký tự thứ `13` là `x`.

In [35]:
s = "xy2z3"
lens, L = [], 0
for ch in s:
    L = L * int(ch) if ch.isdigit() else L + 1
    lens.append(L)
assert lens == [1, 2, 4, 5, 15]
assert decode_all(s) == "xyxyzxyxyzxyxyz" and oracle(s, 13) == "x" == Solution().decodeAtIndex(s, 13)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [36]:
def reverse_answer(s, k):
    L = 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
    for i in range(len(s) - 1, -1, -1):
        k %= L
        if s[i].isdigit():
            L //= int(s[i])
        elif k == 0:
            return s[i]
        else:
            L -= 1
    return ""


assert [reverse_answer(s, k) for (s, k), _ in examples] == ["o", "h", "a"]
rng = random.Random(3)
for s, k in tests[:200]:
    assert reverse_answer(s, k) == oracle(s, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def decodeAtIndex(self, s, k):
        lens, L = [], 0
        for ch in s:
            L = L * int(ch) if ch.isdigit() else L + 1
            lens.append(L)
        while True:
            i = bisect_left(lens, k)
            if not s[i].isdigit():
                return s[i]
            k = (k - 1) % lens[i - 1] + 1


assert all(MySolutionAnswer().decodeAtIndex(s, k) == w for (s, k), w in examples)
for s, k in tests[:200]:
    assert MySolutionAnswer().decodeAtIndex(s, k) == oracle(s, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def source_index_answer(s, k):
    lens, L = [], 0
    for ch in s:
        L = L * int(ch) if ch.isdigit() else L + 1
        lens.append(L)
    while True:
        i = bisect_left(lens, k)
        if not s[i].isdigit():
            return i
        k = (k - 1) % lens[i - 1] + 1


assert source_index_answer("leet2code3", 10) == 6
for s, k in tests[:300]:
    i = source_index_answer(s, k)
    assert s[i] == oracle(s, k) and not s[i].isdigit()
print("Bài 5 OK: s[source_index] = ký tự thứ k")

Bài 5 OK: s[source_index] = ký tự thứ k


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chuỗi nhân bản khổng lồ, hỏi một vị trí → độ dài rồi duyệt ngược):**

```python
class Solution:
    def decodeAtIndex(self, s: str, k: int) -> str:
        length, end = 0, 0
        for i, ch in enumerate(s):
            length = length * int(ch) if ch.isdigit() else length + 1
            if length >= k:
                end = i
                break
        for i in range(end, -1, -1):
            k %= length
            if s[i].isdigit():
                length //= int(s[i])
            elif k == 0:
                return s[i]
            else:
                length -= 1
```

**Checklist trước khi submit:**

- [ ] Không dựng chuỗi giải mã (chỉ lưu độ dài)?
- [ ] Duyệt **ngược**; `k %= length` ở đầu mỗi bước?
- [ ] Chữ số `d`: `length //= d`; chữ cái: trả khi `k == 0`, ngược lại `length -= 1`?
- [ ] Chỉ trả khi `s[i]` là **chữ cái**?
- [ ] `k` 1-indexed; dừng pha xuôi khi `length >= k` (tránh số quá lớn ở ngôn ngữ khác)?
- [ ] Đã thử `leet2code3` với `k = 10`, `ha22` với `k = 5`, `a2345678999999999999999` với `k = 1`, `k` = cuối băng?
- [ ] Đúng tên `class Solution`, `decodeAtIndex(self, s, k)`, không in debug.